In [ ]:
# -*- coding: utf-8 -*-
"""
Single-matrix figure (no facets) with Science-style aesthetics.
- Rows = Experimental Uptakes (substrates) grouped by Classification, then Experimental Secretions (products) grouped by Classification.
- Columns = species (iteration-2 order).
- Shapes: squares = Uptake, circles = Secretion.
- Colors (Okabe–Ito, colorblind-safe):
    BOTH iterations   -> blue  (#0072B2)
    ONLY 1st iteration -> orange (#E69F00)
    ONLY 2nd iteration -> purple (#CC79A7)
- Secretions with Type != "Exact":
    * Row label gets a star (★)
    * Per-species presence accepts ID OR any listed Precursor IDs
    * Cell is RINGED (no fill) if the match is precursor-only in all contributing iterations.
- Classification bands (alternating shading) and class labels on the left.
"""

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures
# ----------------------------
# Paths (EDIT THESE AS NEEDED)
# ----------------------------
ITER2_SUMMARY_PATH = "./summaryAll100_2ndIteration.xlsx"
ITER1_SUMMARY_PATH = "./summaryAll100_1stIteration.xlsx"
EXPERIMENTAL_PATH  = "./ExperimentalValidation.xlsx"
CLASS_PATH         = "./CompoundClassification.xlsx"
OUTPUT_PDF         = "./Species_SingleMatrix_ScienceStyle100.pdf"

# Optional: known alias corrections for IDs (e.g., fix typos to correct KEGG)
# Example: {"C02693": "C00463"}  # indole fix, if needed
ID_ALIASES = {}

# ----------------------------
# Science/Nature-style defaults
# ----------------------------
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "font.size": 9,
})

# Colors (Okabe–Ito, CVD-safe)
COLOR_BOTH   = "#0072B2"  # blue
COLOR_FIRST  = "#E69F00"  # orange (present ONLY in 1st iteration)
COLOR_SECOND = "#CC79A7"  # purple (present ONLY in 2nd iteration)
DIVIDER_COLOR = "#e41a1c"
ROW_BAND     = "#F2F2F2"
MARKER_SIZE  = 80

# ----------------------------
# Helpers
# ----------------------------
def norm_cols(df: pd.DataFrame) -> dict:
    return {str(c).strip().lower().replace(" ", "").replace("_", ""): c for c in df.columns}

def clean_text(x) -> str:
    s = "" if pd.isna(x) else str(x)
    return s.replace("\xa0", " ").strip()

def clean_id(x) -> str:
    s = clean_text(x)
    return ID_ALIASES.get(s, s)

def split_ids(cell) -> list:
    s = clean_text(cell)
    if not s:
        return []
    s = re.sub(r"[;,|/]+", " ", s)
    return [clean_id(p) for p in s.split() if p.strip()]

def load_species_sets(xls_path: str):
    xls = pd.ExcelFile(xls_path)
    sheets = [s for s in xls.sheet_names if "_summary" in s]
    subs_sets, prods_sets = {}, {}
    for s in sheets:
        df = pd.read_excel(xls, sheet_name=s)
        subs_sets[s] = set(clean_id(x) for x in df["substrates"].dropna().astype(str))
        prods_sets[s] = set(clean_id(x) for x in df["products"].dropna().astype(str))
    return sheets, subs_sets, prods_sets

def decide_color(in1: bool, in2: bool):
    """Return the color for BOTH / 1st-only / 2nd-only; None if absent in both."""
    if not (in1 or in2):
        return None
    if in1 and in2:
        return COLOR_BOTH
    return COLOR_FIRST if in1 else COLOR_SECOND

def scatter_cell(ax, x, y, shape: str, facecolor: str, ring_only: bool = False):
    """
    Draw one matrix cell:
      shape='square' (uptake) or 'circle' (secretion)
      facecolor: the decided iteration color
      ring_only=True => draw a ring (no fill), e.g. precursor-only secretions
    """
    if facecolor is None:
        return
    if shape == "square":
        ax.scatter(x, y,
                   marker='s',
                   s=MARKER_SIZE,
                   facecolors="none" if ring_only else facecolor,
                   edgecolors=facecolor if ring_only else "black",
                   linewidths=0.9)
    else:  # circle
        ax.scatter(x, y,
                   marker='o',
                   s=MARKER_SIZE,
                   facecolors="none" if ring_only else facecolor,
                   edgecolors=facecolor if ring_only else "black",
                   linewidths=0.9)

def build_groups(ids_list, class_map, name_map):
    """Return list of (class, [(name, id), ...]) sorted by class then name (case-insensitive)."""
    pairs = [(class_map.get(i, "Unknown"), name_map.get(i, i), i) for i in ids_list]
    pairs.sort(key=lambda x: (x[0].lower(), x[1].lower()))
    groups = {}
    for cls, nm, mid in pairs:
        groups.setdefault(cls, []).append((nm, mid))
    return list(groups.items())

# ----------------------------
# Load classification (Graphing sheet)
# ----------------------------
g = pd.read_excel(CLASS_PATH, sheet_name="Graphing")
g_cols = norm_cols(g)
id_col    = g_cols.get("keggid") or g_cols.get("kegg") or g_cols.get("id")
name_col  = g_cols.get("name")
class_col = g_cols.get("classification")
if not (id_col and class_col):
    raise ValueError("Graphing sheet must include 'KEGG ID' and 'Classification' columns (case-insensitive).")

class_map = {clean_id(k): clean_text(c) for k, c in zip(g[id_col].astype(str), g[class_col].astype(str))}
name_map  = {clean_id(k): clean_text(n) for k, n in zip(g[id_col].astype(str), g[name_col].astype(str))}

# ----------------------------
# Load species profiles (iteration 2 & 1)
# ----------------------------
iter2_sheets, iter2_subs, iter2_prods = load_species_sets(ITER2_SUMMARY_PATH)
iter1_sheets, iter1_subs, iter1_prods = load_species_sets(ITER1_SUMMARY_PATH)
species_labels = [s.replace("_summary", "") for s in iter2_sheets]

# ----------------------------
# Load experimental data
# ----------------------------
exp_xls = pd.ExcelFile(EXPERIMENTAL_PATH)
sec_sheet = "ExperimentalSecretions" if "ExperimentalSecretions" in exp_xls.sheet_names else "ExperiemntalSecretions"
upt_sheet = "ExperimentalUptakes"

sec_df = pd.read_excel(exp_xls, sheet_name=sec_sheet)
upt_df = pd.read_excel(exp_xls, sheet_name=upt_sheet)

# Normalize experimental columns
sec_map = norm_cols(sec_df)
upt_map = norm_cols(upt_df)

sec_id_col   = sec_map["id"];    sec_name_col = sec_map.get("name", sec_map["id"])
sec_type_col = sec_map.get("type")
sec_prec_col = sec_map.get("precursor")

upt_id_col   = upt_map["id"];    upt_name_col = upt_map.get("name", upt_map["id"])

# Build lists (and restrict to compounds present in Graphing so classification is defined)
exp_upt_ids_all = [clean_id(x) for x in upt_df[upt_id_col].tolist()]
exp_sec_ids_all = [clean_id(x) for x in sec_df[sec_id_col].tolist()]

exp_upt_ids = [i for i in exp_upt_ids_all if i in class_map]
exp_sec_ids = [i for i in exp_sec_ids_all if i in class_map]

# Names prefer Graphing sheet for consistency
exp_upt_names = [name_map.get(i, i) for i in exp_upt_ids]
exp_sec_names = [name_map.get(i, i) for i in exp_sec_ids]

# Secretions metadata aligned to exp_sec_ids
sec_df_idx = sec_df.set_index(sec_id_col)
exp_sec_types = [clean_text(sec_df_idx.loc[i, sec_type_col]).lower() if (sec_type_col and i in sec_df_idx.index) else "" for i in exp_sec_ids]
if sec_prec_col:
    exp_sec_prec = [split_ids(sec_df_idx.loc[i, sec_prec_col]) if i in sec_df_idx.index else [] for i in exp_sec_ids]
else:
    exp_sec_prec = [[] for _ in exp_sec_ids]
star_flags = [(t != "exact") and (t != "") for t in exp_sec_types]

# ----------------------------
# Group by classification within each block
# ----------------------------
upt_groups = build_groups(exp_upt_ids, class_map, name_map)  # list of (class, [(name, id), ...])
sec_groups = build_groups(exp_sec_ids, class_map, name_map)

# Build row index, labels, and class boundaries for uptakes
upt_row_ids, upt_row_labels, upt_class_bounds = [], [], []
row_idx = 0
for cls, items in upt_groups:
    start = row_idx
    for nm, mid in items:
        upt_row_ids.append(mid)
        upt_row_labels.append(nm)
        row_idx += 1
    upt_class_bounds.append((start, row_idx - 1, cls))

# Build row index, labels, and class boundaries for secretions
sec_row_ids, sec_row_labels, sec_class_bounds = [], [], []
row_idx = 0
for cls, items in sec_groups:
    start = row_idx
    for nm, mid in items:
        idx = exp_sec_ids.index(mid)
        label = nm + (" ★" if star_flags[idx] else "")
        sec_row_ids.append(mid)
        sec_row_labels.append(label)
        row_idx += 1
    sec_class_bounds.append((start, row_idx - 1, cls))

# ----------------------------
# Plot single matrix
# ----------------------------
fig, ax = plt.subplots(figsize=(16, 24))

# Uptakes block (squares)
for i, mid in enumerate(upt_row_ids):
    for j, sheet in enumerate(iter2_sheets):
        # presence per iteration
        in2 = (mid in iter2_subs[sheet])
        in1 = (mid in iter1_subs.get(sheet, set()))
        color = decide_color(in1, in2)
        scatter_cell(ax, j, i, "square", color, ring_only=False)

# Secretions block (circles), offset by number of uptake rows
offset = len(upt_row_ids)
for i, mid in enumerate(sec_row_ids):
    idx = exp_sec_ids.index(mid)
    is_star = star_flags[idx]
    precs   = exp_sec_prec[idx]

    for j, sheet in enumerate(iter2_sheets):
        # exact ID / precursor flags per iteration
        in2_id   = (mid in iter2_prods[sheet])
        in2_prec = (is_star and any(p in iter2_prods[sheet] for p in precs))
        in1_id   = (mid in iter1_prods.get(sheet, set()))
        in1_prec = (is_star and any(p in iter1_prods.get(sheet, set()) for p in precs))

        in2 = in2_id or in2_prec
        in1 = in1_id or in1_prec

        color = decide_color(in1, in2)
        # ring if present AND for every contributing iteration the exact-ID flag is False
        ring_only = False
        if in1 or in2:
            contrib_id_flags = []
            if in1: contrib_id_flags.append(in1_id)
            if in2: contrib_id_flags.append(in2_id)
            ring_only = all(f is False for f in contrib_id_flags)

        scatter_cell(ax, j, offset + i, "circle", color, ring_only=ring_only)

# Divider between Uptake and Secretion blocks
ax.axhline(y=offset - 0.5, color=DIVIDER_COLOR, linewidth=2.0, linestyle="--")

# Alternating row shading by classification blocks + class labels at left
def shade_and_label(bounds, y_offset=0):
    for k, (start, end, cls) in enumerate(bounds):
        if start <= end:
            # alternating banding per class
            if k % 2 == 0:
                ax.axhspan(y_offset + start - 0.5, y_offset + end + 0.5, color=ROW_BAND, alpha=0.6, lw=0)
            # class label at left margin
            y_mid = y_offset + (start + end) / 2.0
            ax.text(-0.8, y_mid, cls, va="center", ha="right", fontsize=9, fontweight="bold")

shade_and_label(upt_class_bounds, y_offset=0)
shade_and_label(sec_class_bounds, y_offset=offset)

# Axes / labels
ax.set_title("Experimental Uptakes & Secretions by Species — Grouped by Classification\n"
             "Color: BOTH (blue), 1st-only (orange), 2nd-only (purple)  |  Shape: Uptake (■), Secretion (●)  |  Ring: precursor-only",
             fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, len(iter2_sheets) - 0.5)
ax.set_xticks(range(len(iter2_sheets)))
ax.set_xticklabels(species_labels, rotation=90, fontsize=9, fontweight='bold')

yticks = list(range(len(upt_row_labels) + len(sec_row_labels)))
yticklabels = upt_row_labels + sec_row_labels
ax.set_ylim(-0.5, len(yticks) - 0.5)
ax.set_yticks(yticks)
ax.set_yticklabels(yticklabels, fontsize=9)

ax.grid(axis='x', color='#E0E0E0', linestyle=':', linewidth=0.6)
ax.set_axisbelow(True)

# Legend
both_patch   = plt.Line2D([0],[0], marker='s', linestyle='', markerfacecolor=COLOR_BOTH,   markeredgecolor='black', markersize=8)
first_patch  = plt.Line2D([0],[0], marker='s', linestyle='', markerfacecolor=COLOR_FIRST,  markeredgecolor='black', markersize=8)
second_patch = plt.Line2D([0],[0], marker='s', linestyle='', markerfacecolor=COLOR_SECOND, markeredgecolor='black', markersize=8)
upt_proxy    = plt.Line2D([0],[0], marker='s', linestyle='', markerfacecolor='white', markeredgecolor='black', markersize=8)
sec_proxy    = plt.Line2D([0],[0], marker='o', linestyle='', markerfacecolor='white', markeredgecolor='black', markersize=8)
ring_proxy   = plt.Line2D([0],[0], marker='o', linestyle='', markerfacecolor='none', markeredgecolor=COLOR_BOTH, markersize=8)

legend = ax.legend(
    [both_patch, first_patch, second_patch, upt_proxy, sec_proxy, ring_proxy],
    ['Present in BOTH iterations', 'Present ONLY in 1st', 'Present ONLY in 2nd', 'Uptake marker (shape)', 'Secretion marker (shape)', 'Precursor-only (ring)'],
    loc='upper left', fontsize=9, frameon=True, fancybox=True
)
legend.get_frame().set_edgecolor('black')
legend.get_frame().set_linewidth(1.2)

plt.tight_layout()
plt.savefig(OUTPUT_PDF, dpi=300)
plt.show()

print(f"Saved figure: {OUTPUT_PDF}")


In [ ]:
# -*- coding: utf-8 -*-
"""
Single-matrix figure (no facets) with Science-style aesthetics.
- Rows = Experimental Uptakes (substrates) grouped by Classification, then Experimental Secretions (products) grouped by Classification.
- Columns = species (iteration-2 order).
- Shapes: squares = Uptake, circles = Secretion.
- Colors (Okabe–Ito, colorblind-safe):
    BOTH iterations   -> blue  (#0072B2)
    ONLY 1st iteration -> orange (#E69F00)
    ONLY 2nd iteration -> purple (#CC79A7)
- Secretions with Type != "Exact":
    * Row label gets a star (★)
    * Per-species presence accepts ID OR any listed Precursor IDs
    * Cell is RINGED (no fill) if the match is precursor-only in all contributing iterations.
- Classification bands (alternating shading) and class labels on the left.
"""

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures
# ----------------------------
# Paths (EDIT THESE AS NEEDED)
# ----------------------------
ITER2_SUMMARY_PATH = "./summaryAll2ndIteration.xlsx"
ITER1_SUMMARY_PATH = "./summaryAll1stIteration.xlsx"
EXPERIMENTAL_PATH  = "./ExperimentalValidation.xlsx"
CLASS_PATH         = "./CompoundClassification.xlsx"
OUTPUT_PDF         = "Species_SingleMatrix_ScienceStyleUnion.pdf"

# Optional: known alias corrections for IDs (e.g., fix typos to correct KEGG)
# Example: {"C02693": "C00463"}  # indole fix, if needed
ID_ALIASES = {}

# ----------------------------
# Science/Nature-style defaults
# ----------------------------
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "font.size": 9,
})

# Colors (Okabe–Ito, CVD-safe)
COLOR_BOTH   = "#0072B2"  # blue
COLOR_FIRST  = "#E69F00"  # orange (present ONLY in 1st iteration)
COLOR_SECOND = "#CC79A7"  # purple (present ONLY in 2nd iteration)
DIVIDER_COLOR = "#e41a1c"
ROW_BAND     = "#F2F2F2"
MARKER_SIZE  = 80

# ----------------------------
# Helpers
# ----------------------------
def norm_cols(df: pd.DataFrame) -> dict:
    return {str(c).strip().lower().replace(" ", "").replace("_", ""): c for c in df.columns}

def clean_text(x) -> str:
    s = "" if pd.isna(x) else str(x)
    return s.replace("\xa0", " ").strip()

def clean_id(x) -> str:
    s = clean_text(x)
    return ID_ALIASES.get(s, s)

def split_ids(cell) -> list:
    s = clean_text(cell)
    if not s:
        return []
    s = re.sub(r"[;,|/]+", " ", s)
    return [clean_id(p) for p in s.split() if p.strip()]

def load_species_sets(xls_path: str):
    xls = pd.ExcelFile(xls_path)
    sheets = [s for s in xls.sheet_names if "_summary" in s]
    subs_sets, prods_sets = {}, {}
    for s in sheets:
        df = pd.read_excel(xls, sheet_name=s)
        subs_sets[s] = set(clean_id(x) for x in df["substrates"].dropna().astype(str))
        prods_sets[s] = set(clean_id(x) for x in df["products"].dropna().astype(str))
    return sheets, subs_sets, prods_sets

def decide_color(in1: bool, in2: bool):
    """Return the color for BOTH / 1st-only / 2nd-only; None if absent in both."""
    if not (in1 or in2):
        return None
    if in1 and in2:
        return COLOR_BOTH
    return COLOR_FIRST if in1 else COLOR_SECOND

def scatter_cell(ax, x, y, shape: str, facecolor: str, ring_only: bool = False):
    """
    Draw one matrix cell:
      shape='square' (uptake) or 'circle' (secretion)
      facecolor: the decided iteration color
      ring_only=True => draw a ring (no fill), e.g. precursor-only secretions
    """
    if facecolor is None:
        return
    if shape == "square":
        ax.scatter(x, y,
                   marker='s',
                   s=MARKER_SIZE,
                   facecolors="none" if ring_only else facecolor,
                   edgecolors=facecolor if ring_only else "black",
                   linewidths=0.9)
    else:  # circle
        ax.scatter(x, y,
                   marker='o',
                   s=MARKER_SIZE,
                   facecolors="none" if ring_only else facecolor,
                   edgecolors=facecolor if ring_only else "black",
                   linewidths=0.9)

def build_groups(ids_list, class_map, name_map):
    """Return list of (class, [(name, id), ...]) sorted by class then name (case-insensitive)."""
    pairs = [(class_map.get(i, "Unknown"), name_map.get(i, i), i) for i in ids_list]
    pairs.sort(key=lambda x: (x[0].lower(), x[1].lower()))
    groups = {}
    for cls, nm, mid in pairs:
        groups.setdefault(cls, []).append((nm, mid))
    return list(groups.items())

# ----------------------------
# Load classification (Graphing sheet)
# ----------------------------
g = pd.read_excel(CLASS_PATH, sheet_name="Graphing")
g_cols = norm_cols(g)
id_col    = g_cols.get("keggid") or g_cols.get("kegg") or g_cols.get("id")
name_col  = g_cols.get("name")
class_col = g_cols.get("classification")
if not (id_col and class_col):
    raise ValueError("Graphing sheet must include 'KEGG ID' and 'Classification' columns (case-insensitive).")

class_map = {clean_id(k): clean_text(c) for k, c in zip(g[id_col].astype(str), g[class_col].astype(str))}
name_map  = {clean_id(k): clean_text(n) for k, n in zip(g[id_col].astype(str), g[name_col].astype(str))}

# ----------------------------
# Load species profiles (iteration 2 & 1)
# ----------------------------
iter2_sheets, iter2_subs, iter2_prods = load_species_sets(ITER2_SUMMARY_PATH)
iter1_sheets, iter1_subs, iter1_prods = load_species_sets(ITER1_SUMMARY_PATH)
species_labels = [s.replace("_summary", "") for s in iter2_sheets]

# ----------------------------
# Load experimental data
# ----------------------------
exp_xls = pd.ExcelFile(EXPERIMENTAL_PATH)
sec_sheet = "ExperimentalSecretions" if "ExperimentalSecretions" in exp_xls.sheet_names else "ExperiemntalSecretions"
upt_sheet = "ExperimentalUptakes"

sec_df = pd.read_excel(exp_xls, sheet_name=sec_sheet)
upt_df = pd.read_excel(exp_xls, sheet_name=upt_sheet)

# Normalize experimental columns
sec_map = norm_cols(sec_df)
upt_map = norm_cols(upt_df)

sec_id_col   = sec_map["id"];    sec_name_col = sec_map.get("name", sec_map["id"])
sec_type_col = sec_map.get("type")
sec_prec_col = sec_map.get("precursor")

upt_id_col   = upt_map["id"];    upt_name_col = upt_map.get("name", upt_map["id"])

# Build lists (and restrict to compounds present in Graphing so classification is defined)
exp_upt_ids_all = [clean_id(x) for x in upt_df[upt_id_col].tolist()]
exp_sec_ids_all = [clean_id(x) for x in sec_df[sec_id_col].tolist()]

exp_upt_ids = [i for i in exp_upt_ids_all if i in class_map]
exp_sec_ids = [i for i in exp_sec_ids_all if i in class_map]

# Names prefer Graphing sheet for consistency
exp_upt_names = [name_map.get(i, i) for i in exp_upt_ids]
exp_sec_names = [name_map.get(i, i) for i in exp_sec_ids]

# Secretions metadata aligned to exp_sec_ids
sec_df_idx = sec_df.set_index(sec_id_col)
exp_sec_types = [clean_text(sec_df_idx.loc[i, sec_type_col]).lower() if (sec_type_col and i in sec_df_idx.index) else "" for i in exp_sec_ids]
if sec_prec_col:
    exp_sec_prec = [split_ids(sec_df_idx.loc[i, sec_prec_col]) if i in sec_df_idx.index else [] for i in exp_sec_ids]
else:
    exp_sec_prec = [[] for _ in exp_sec_ids]
star_flags = [(t != "exact") and (t != "") for t in exp_sec_types]

# ----------------------------
# Group by classification within each block
# ----------------------------
upt_groups = build_groups(exp_upt_ids, class_map, name_map)  # list of (class, [(name, id), ...])
sec_groups = build_groups(exp_sec_ids, class_map, name_map)

# Build row index, labels, and class boundaries for uptakes
upt_row_ids, upt_row_labels, upt_class_bounds = [], [], []
row_idx = 0
for cls, items in upt_groups:
    start = row_idx
    for nm, mid in items:
        upt_row_ids.append(mid)
        upt_row_labels.append(nm)
        row_idx += 1
    upt_class_bounds.append((start, row_idx - 1, cls))

# Build row index, labels, and class boundaries for secretions
sec_row_ids, sec_row_labels, sec_class_bounds = [], [], []
row_idx = 0
for cls, items in sec_groups:
    start = row_idx
    for nm, mid in items:
        idx = exp_sec_ids.index(mid)
        label = nm + (" ★" if star_flags[idx] else "")
        sec_row_ids.append(mid)
        sec_row_labels.append(label)
        row_idx += 1
    sec_class_bounds.append((start, row_idx - 1, cls))

# ----------------------------
# Plot single matrix
# ----------------------------
fig, ax = plt.subplots(figsize=(16, 24))

# Uptakes block (squares)
for i, mid in enumerate(upt_row_ids):
    for j, sheet in enumerate(iter2_sheets):
        # presence per iteration
        in2 = (mid in iter2_subs[sheet])
        in1 = (mid in iter1_subs.get(sheet, set()))
        color = decide_color(in1, in2)
        scatter_cell(ax, j, i, "square", color, ring_only=False)

# Secretions block (circles), offset by number of uptake rows
offset = len(upt_row_ids)
for i, mid in enumerate(sec_row_ids):
    idx = exp_sec_ids.index(mid)
    is_star = star_flags[idx]
    precs   = exp_sec_prec[idx]

    for j, sheet in enumerate(iter2_sheets):
        # exact ID / precursor flags per iteration
        in2_id   = (mid in iter2_prods[sheet])
        in2_prec = (is_star and any(p in iter2_prods[sheet] for p in precs))
        in1_id   = (mid in iter1_prods.get(sheet, set()))
        in1_prec = (is_star and any(p in iter1_prods.get(sheet, set()) for p in precs))

        in2 = in2_id or in2_prec
        in1 = in1_id or in1_prec

        color = decide_color(in1, in2)
        # ring if present AND for every contributing iteration the exact-ID flag is False
        ring_only = False
        if in1 or in2:
            contrib_id_flags = []
            if in1: contrib_id_flags.append(in1_id)
            if in2: contrib_id_flags.append(in2_id)
            ring_only = all(f is False for f in contrib_id_flags)

        scatter_cell(ax, j, offset + i, "circle", color, ring_only=ring_only)

# Divider between Uptake and Secretion blocks
ax.axhline(y=offset - 0.5, color=DIVIDER_COLOR, linewidth=2.0, linestyle="--")

# Alternating row shading by classification blocks + class labels at left
def shade_and_label(bounds, y_offset=0):
    for k, (start, end, cls) in enumerate(bounds):
        if start <= end:
            # alternating banding per class
            if k % 2 == 0:
                ax.axhspan(y_offset + start - 0.5, y_offset + end + 0.5, color=ROW_BAND, alpha=0.6, lw=0)
            # class label at left margin
            y_mid = y_offset + (start + end) / 2.0
            ax.text(-0.8, y_mid, cls, va="center", ha="right", fontsize=9, fontweight="bold")

shade_and_label(upt_class_bounds, y_offset=0)
shade_and_label(sec_class_bounds, y_offset=offset)

# Axes / labels
ax.set_title("Experimental Uptakes & Secretions by Species — Grouped by Classification\n"
             "Color: BOTH (blue), 1st-only (orange), 2nd-only (purple)  |  Shape: Uptake (■), Secretion (●)  |  Ring: precursor-only",
             fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, len(iter2_sheets) - 0.5)
ax.set_xticks(range(len(iter2_sheets)))
ax.set_xticklabels(species_labels, rotation=90, fontsize=9, fontweight='bold')

yticks = list(range(len(upt_row_labels) + len(sec_row_labels)))
yticklabels = upt_row_labels + sec_row_labels
ax.set_ylim(-0.5, len(yticks) - 0.5)
ax.set_yticks(yticks)
ax.set_yticklabels(yticklabels, fontsize=9)

ax.grid(axis='x', color='#E0E0E0', linestyle=':', linewidth=0.6)
ax.set_axisbelow(True)

# Legend
both_patch   = plt.Line2D([0],[0], marker='s', linestyle='', markerfacecolor=COLOR_BOTH,   markeredgecolor='black', markersize=8)
first_patch  = plt.Line2D([0],[0], marker='s', linestyle='', markerfacecolor=COLOR_FIRST,  markeredgecolor='black', markersize=8)
second_patch = plt.Line2D([0],[0], marker='s', linestyle='', markerfacecolor=COLOR_SECOND, markeredgecolor='black', markersize=8)
upt_proxy    = plt.Line2D([0],[0], marker='s', linestyle='', markerfacecolor='white', markeredgecolor='black', markersize=8)
sec_proxy    = plt.Line2D([0],[0], marker='o', linestyle='', markerfacecolor='white', markeredgecolor='black', markersize=8)
ring_proxy   = plt.Line2D([0],[0], marker='o', linestyle='', markerfacecolor='none', markeredgecolor=COLOR_BOTH, markersize=8)

legend = ax.legend(
    [both_patch, first_patch, second_patch, upt_proxy, sec_proxy, ring_proxy],
    ['Present in BOTH iterations', 'Present ONLY in 1st', 'Present ONLY in 2nd', 'Uptake marker (shape)', 'Secretion marker (shape)', 'Precursor-only (ring)'],
    loc='upper left', fontsize=9, frameon=True, fancybox=True
)
legend.get_frame().set_edgecolor('black')
legend.get_frame().set_linewidth(1.2)

plt.tight_layout()
plt.savefig(OUTPUT_PDF, dpi=300)
plt.show()

print(f"Saved figure: {OUTPUT_PDF}")
